# Aplicación web en Gradio — Índice de Vulnerabilidad en Pereira

Proyecto académico de Machine Learning para clasificación binaria de niveles de vulnerabilidad.

**Objetivo del notebook:** construir una aplicación web en Google Colab con dos componentes principales:

1. **Chatbot del proyecto**, restringido a preguntas sobre la base de datos, variables, modelo, métricas, resultados, limitaciones y presentación.
2. **Simulador de clasificación**, donde una persona ingresa un perfil sociodemográfico y el modelo predice `vulnerabilidad_baja` o `vulnerabilidad_alta`.

> Nota ética: esta aplicación es académica y exploratoria. No debe usarse como herramienta real de decisión social, focalización de ayudas o política pública sin validación técnica, institucional y ética adicional.

## 1. Instalación de librerías

Ejecuta esta celda primero. En Colab puede tardar unos segundos.

In [36]:
!pip -q install gradio joblib scikit-learn pandas numpy matplotlib pypdf

## 2. Importar librerías y definir configuración general

Esta celda centraliza imports, rutas y variables esperadas del proyecto.

In [57]:
import os
import re
import glob
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns # Added seaborn import
import joblib
import gradio as gr

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

warnings.filterwarnings("ignore")

# ==============================
# Rutas esperadas en Colab
# ==============================
DATA_PATH = "/dataset_definitivo.csv"
MODEL_PATH = "/modelo_Clasificacion_final.pkl"
METRICS_PATH = "/metricas_modelo.csv"
PDF_PATH = "/Indice_Vulnerabilidad_Pereira.pdf"

# Si estás probando localmente en este entorno, estas rutas alternativas ayudan.
ALT_DATA_PATH = "/mnt/data/dataset_definitivo.csv"
ALT_MODEL_PATH = "/mnt/data/modelo_Clasificacion_final.pkl"
ALT_METRICS_PATH = "/mnt/data/metricas_modelo.csv"
ALT_PDF_PATH = "/mnt/data/Indice_Vulnerabilidad_Pereira.pdf"

# Variables predictoras esperadas para el simulador.
FEATURES = [
    "GENERO",
    "EDAD",
    "PERSONAS_EN_CONDICION_DE_DISCAPACIDAD",
    "HOMBRES_Y_MUJERES_CABEZA_DE_FAMILIA",
    "ADULTO_MAYOR",
    "GRUPOS_ETNICOS_AFRO_INDIGENA",
    "TIPO_DE_SEGURIDAD_EN_SALUD",
    "NIVEL_EDUCATIVO_QUE_TIENE_O_CURSA",
    "CONDICION_OCUPACIONAL",
    "BARRIO_O_VEREDA_DE_RESIDENCIA",
    "ZONA_DE_RESIDENCIA",
]

NUMERIC_FEATURES = ["EDAD"]
ORDINAL_FEATURES = ["NIVEL_EDUCATIVO_QUE_TIENE_O_CURSA"]
ONEHOT_FEATURES = [c for c in FEATURES if c not in NUMERIC_FEATURES + ORDINAL_FEATURES]

TARGET_COL = "ETIQUETA_VULNERABILIDAD"
SCORE_COL = "PUNTAJE_VULNERABILIDAD"

print("Configuración cargada correctamente.")

Configuración cargada correctamente.


## 3. Subir o ubicar archivos del proyecto

Sube a Colab, desde el panel izquierdo o con el botón de carga, estos archivos:

- `dataset_definitivo.csv`
- `modelo_Clasificacion_final.pkl` opcional, si está disponible
- `metricas_modelo.csv` opcional
- `Indice_Vulnerabilidad_Pereira.pdf` opcional para alimentar el chatbot

Si los archivos ya están en `/content`, no tienes que hacer nada adicional.

In [38]:
# Opción manual de carga. Puedes descomentar estas líneas si prefieres cargar desde el navegador.
# from google.colab import files
# uploaded = files.upload()

# Función auxiliar para resolver rutas en Colab o entorno local.
def resolve_path(primary_path, alt_path=None):
    if os.path.exists(primary_path):
        return primary_path
    if alt_path and os.path.exists(alt_path):
        return alt_path
    return primary_path

DATA_PATH = resolve_path(DATA_PATH, ALT_DATA_PATH)
MODEL_PATH = resolve_path(MODEL_PATH, ALT_MODEL_PATH)
METRICS_PATH = resolve_path(METRICS_PATH, ALT_METRICS_PATH)
PDF_PATH = resolve_path(PDF_PATH, ALT_PDF_PATH)

print("Ruta CSV:", DATA_PATH, "| Existe:", os.path.exists(DATA_PATH))
print("Ruta modelo:", MODEL_PATH, "| Existe:", os.path.exists(MODEL_PATH))
print("Ruta métricas:", METRICS_PATH, "| Existe:", os.path.exists(METRICS_PATH))
print("Ruta PDF:", PDF_PATH, "| Existe:", os.path.exists(PDF_PATH))

Ruta CSV: /dataset_definitivo.csv | Existe: True
Ruta modelo: /modelo_Clasificacion_final.pkl | Existe: True
Ruta métricas: /metricas_modelo.csv | Existe: True
Ruta PDF: /Indice_Vulnerabilidad_Pereira.pdf | Existe: True


## 4. Cargar y preparar el dataset

La variable objetivo se reconstruye con la **mediana de `PUNTAJE_VULNERABILIDAD`** cuando `ETIQUETA_VULNERABILIDAD` no existe. Además, se elimina `PUNTAJE_VULNERABILIDAD` del conjunto predictor para evitar fuga de información.

In [39]:
def normalize_column_names(df):
    """Normaliza nombres de columnas para reducir errores por espacios o tildes."""
    df = df.copy()
    df.columns = (
        df.columns.astype(str)
        .str.strip()
        .str.upper()
        .str.replace(" ", "_", regex=False)
        .str.replace("Á", "A", regex=False)
        .str.replace("É", "E", regex=False)
        .str.replace("Í", "I", regex=False)
        .str.replace("Ó", "O", regex=False)
        .str.replace("Ú", "U", regex=False)
        .str.replace("Ñ", "N", regex=False)
    )
    return df


def normalize_text_values(df):
    """Normaliza valores categóricos tipo texto."""
    df = df.copy()
    for col in df.select_dtypes(include=["object", "category"]).columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .str.replace(r"\s+", " ", regex=True)
        )
        df[col] = df[col].replace({"NAN": np.nan, "NONE": np.nan, "": np.nan})
    return df


def build_target_with_median(df):
    """Crea ETIQUETA_VULNERABILIDAD usando la mediana del puntaje."""
    df = df.copy()
    if SCORE_COL not in df.columns:
        raise ValueError(f"No existe la columna {SCORE_COL}. No se puede reconstruir la etiqueta.")

    df[SCORE_COL] = pd.to_numeric(df[SCORE_COL], errors="coerce")
    median_score = df[SCORE_COL].median()

    # Regla pedida para la app: corte por mediana.
    df[TARGET_COL] = np.where(
        df[SCORE_COL] > median_score,
        "vulnerabilidad_alta",
        "vulnerabilidad_baja"
    )
    return df, median_score


def load_dataset(path):
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"No encontré el archivo CSV en {path}. Sube dataset_definitivo.csv a /content."
        )

    df = pd.read_csv(path)
    df = normalize_column_names(df)
    df = normalize_text_values(df)

    # Ajuste de nombres por si vienen con variaciones.
    rename_map = {
        "PERSONAS_EN_CONDICIÓN_DE_DISCAPACIDAD": "PERSONAS_EN_CONDICION_DE_DISCAPACIDAD",
        "TIPO_DE_SEGURIDAD_EN_SALUD": "TIPO_DE_SEGURIDAD_EN_SALUD",
        "GRUPOS_ETNICOS_AFRO/INDIGENA": "GRUPOS_ETNICOS_AFRO_INDIGENA",
        "GRUPOS_ETNICOS_AFRO_INDÍGENA": "GRUPOS_ETNICOS_AFRO_INDIGENA",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    if TARGET_COL not in df.columns:
        df, median_score = build_target_with_median(df)
    else:
        median_score = df[SCORE_COL].median() if SCORE_COL in df.columns else None
        df[TARGET_COL] = df[TARGET_COL].astype(str).str.lower().str.strip()

    # Asegurar edad numérica.
    if "EDAD" in df.columns:
        df["EDAD"] = pd.to_numeric(df["EDAD"], errors="coerce")

    missing_features = [c for c in FEATURES if c not in df.columns]
    if missing_features:
        raise ValueError(f"Faltan variables esperadas en el CSV: {missing_features}")

    # Limpiar filas sin datos esenciales.
    df = df.dropna(subset=FEATURES + [TARGET_COL]).reset_index(drop=True)

    return df, median_score


df, median_score = load_dataset(DATA_PATH)

print("Dataset cargado:", df.shape)
print("Mediana usada para la etiqueta:", median_score)
print("Distribución de la etiqueta:")
print(df[TARGET_COL].value_counts())
print("\nPrimeras filas:")
display(df.head())

Dataset cargado: (13987, 15)
Mediana usada para la etiqueta: 3.0
Distribución de la etiqueta:
ETIQUETA_VULNERABILIDAD
vulnerabilidad_baja    9635
vulnerabilidad_alta    4352
Name: count, dtype: int64

Primeras filas:


,GENERO,EDAD,PERSONAS_EN_CONDICION_DE_DISCAPACIDAD,HOMBRES_Y_MUJERES_CABEZA_DE_FAMILIA,ADULTO_MAYOR,GRUPOS_ETNICOS_AFRO_INDIGENA,TIPO_DE_SEGURIDAD_EN_SALUD,NIVEL_EDUCATIVO_QUE_TIENE_O_CURSA,CONDICION_OCUPACIONAL,BARRIO_O_VEREDA_DE_RESIDENCIA,ZONA_DE_RESIDENCIA,PUNTAJE_VULNERABILIDAD,NIVEL_VULNERABILIDAD,RANGO_EDAD,ETIQUETA_VULNERABILIDAD
0,F,19,NO,NO,NO,6 MESTIZO,REGIMEN CONTRIBUTIVO,8- TECNOLÓGICA,BUSCANDO EMPLEO,LOS 2500,URBANO,3,ALTO,16-25,vulnerabilidad_baja
1,F,30,NO,NO,NO,6 MESTIZO,REGIMEN SUBSIDIADO,2- BÁSICA PRIMARIA,INDEPENDIENTE,OTRO MUNICIPIO,OTRO,4,ALTO,26-35,vulnerabilidad_alta
2,M,43,NO,NO,NO,6 MESTIZO,REGIMEN CONTRIBUTIVO,12- DOCTORADO,EMPLEADO,CUBA,URBANO,0,BAJO,36-45,vulnerabilidad_baja
3,M,65,NO,NO,SI,6 MESTIZO,REGIMEN CONTRIBUTIVO,3- BÁSICA SECUNDARIA,EMPLEADO,CUBA,URBANO,1,BAJO,56-65,vulnerabilidad_baja
4,M,67,NO,NO,SI,6 MESTIZO,REGIMEN CONTRIBUTIVO,2- BÁSICA PRIMARIA,EMPLEADO,TERRANOVA,URBANO,3,ALTO,66-85,vulnerabilidad_baja


## 5. Crear preprocesamiento y entrenar modelo de respaldo

Este modelo se usa si el `.pkl` no carga por problemas de versión, ruta o compatibilidad. El pipeline incluye el preprocesamiento y el clasificador.

In [40]:
# Orden educativo recomendado. Si aparece un valor no visto, OrdinalEncoder lo codifica como -1.
EDUCATION_ORDER = [[
    "SIN INFORMACION",
    "14- NINGUNO",
    "1- PREESCOLAR",
    "2- BASICA PRIMARIA",
    "2- BÁSICA PRIMARIA",
    "3- BASICA SECUNDARIA",
    "3- BÁSICA SECUNDARIA",
    "4- MEDIA ACADEMICA O CLASICA",
    "4- MEDIA ACADÉMICA O CLÁSICA",
    "5- MEDIA TECNICA (BACHILLERATO TECNICO)",
    "5- MEDIA TÉCNICA (BACHILLERATO TÉCNICO)",
    "6- NORMALISTA",
    "7- TECNICA PROFESIONAL",
    "7- TÉCNICA PROFESIONAL",
    "8- TECNOLOGICA",
    "8- TECNOLÓGICA",
    "13-UNIVERISITARIO",
    "9- PROFESIONAL",
    "10- ESPECIALIZACION",
    "10- ESPECIALIZACIÓN",
    "11- MAESTRIA",
    "11- MAESTRÍA",
    "12- DOCTORADO",
]]


def make_preprocessor():
    ordinal_encoder = OrdinalEncoder(
        categories=EDUCATION_ORDER,
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    onehot_encoder = OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("ordinal_educacion", ordinal_encoder, ORDINAL_FEATURES),
            ("onehot_categoricas", onehot_encoder, ONEHOT_FEATURES),
            ("numerica_edad", "passthrough", NUMERIC_FEATURES),
        ],
        remainder="drop"
    )
    return preprocessor


def train_fallback_pipeline(df):
    """Entrena un pipeline completo desde el CSV."""
    X = df[FEATURES].copy()
    y = df[TARGET_COL].copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )

    pipeline = Pipeline(steps=[
        ("preprocessor", make_preprocessor()),
        ("model", RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1
        ))
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    report = {
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "confusion_matrix": confusion_matrix(y_test, y_pred, labels=["vulnerabilidad_baja", "vulnerabilidad_alta"]).tolist(),
        "classification_report": classification_report(y_test, y_pred, output_dict=True),
    }
    return pipeline, report

print("Funciones de preprocesamiento y entrenamiento listas.")

Funciones de preprocesamiento y entrenamiento listas.


## 6. Cargar el modelo `.pkl` o reentrenar automáticamente

Primero intenta cargar `modelo_Clasificacion_final.pkl`. Si falla por versión de scikit-learn o estructura del archivo, entrena un Random Forest desde el CSV.

In [41]:
def extract_model_from_loaded_object(obj):
    """Extrae el modelo/pipeline si el .pkl viene como objeto directo o diccionario."""
    if hasattr(obj, "predict"):
        return obj, {}
    if isinstance(obj, dict):
        possible_keys = ["model", "modelo", "pipeline", "best_model", "mejor_modelo", "clf"]
        for key in possible_keys:
            if key in obj and hasattr(obj[key], "predict"):
                metadata = {k: v for k, v in obj.items() if k != key}
                return obj[key], metadata
    raise ValueError("El archivo .pkl no contiene un modelo compatible con predict().")


def load_or_train_model(model_path, df):
    load_status = ""
    metadata = {}
    model_report = None

    if os.path.exists(model_path):
        try:
            loaded = joblib.load(model_path)
            model, metadata = extract_model_from_loaded_object(loaded)
            # Prueba rápida de compatibilidad con una fila.
            _ = model.predict(df[FEATURES].head(1))
            load_status = "✅ Modelo .pkl cargado correctamente."
            return model, load_status, metadata, model_report
        except Exception as e:
            load_status = (
                "⚠️ No se pudo cargar el modelo .pkl por compatibilidad, versión o estructura. "
                f"Se entrenará un modelo de respaldo desde el CSV. Detalle: {repr(e)}"
            )
    else:
        load_status = "⚠️ No se encontró modelo .pkl. Se entrenará un modelo de respaldo desde el CSV."

    model, model_report = train_fallback_pipeline(df)
    return model, load_status, metadata, model_report

model, load_status, model_metadata, fallback_report = load_or_train_model(MODEL_PATH, df)

print(load_status)
if fallback_report:
    print("Accuracy del modelo reentrenado:", round(fallback_report["accuracy"], 4))
    print("Matriz de confusión [baja, alta]:")
    print(np.array(fallback_report["confusion_matrix"]))

✅ Modelo .pkl cargado correctamente.


## 7. Preparar catálogos, validaciones y función de predicción

Los desplegables permiten seleccionar valores existentes, pero también escribir nuevos valores. Si se escriben valores no vistos en el dataset, la app muestra una advertencia.

In [42]:
def get_options(df, col):
    vals = sorted([str(x) for x in df[col].dropna().unique()])
    return vals

CATEGORY_OPTIONS = {col: get_options(df, col) for col in FEATURES if col != "EDAD"}
AGE_MIN = int(df["EDAD"].min())
AGE_MAX = int(df["EDAD"].max())

# =========================================================
# DEPURACIÓN IMPORTANTE
# =========================================================
# Algunos modelos .pkl guardan las clases como números:
# 1 = VULNERABILIDAD_BAJA
# 2 = VULNERABILIDAD_ALTA
#
# Si se interpreta "1" automáticamente como alta, la app invierte el resultado.
# Por eso usamos el diccionario de metadatos del .pkl cuando existe.
# =========================================================

def make_age_range(age):
    """Construye RANGO_EDAD automáticamente si el modelo cargado lo necesita."""
    try:
        age = float(age)
    except Exception:
        return "SIN INFORMACION"

    if age <= 25:
        return "16-25"
    elif age <= 35:
        return "26-35"
    elif age <= 45:
        return "36-45"
    elif age <= 55:
        return "46-55"
    elif age <= 65:
        return "56-65"
    else:
        return "66-85"


def get_expected_model_columns(model, metadata):
    """
    Devuelve las columnas que espera el modelo.
    Prioridad:
    1. columnas_entrada guardadas en el .pkl
    2. FEATURES definidas para la app
    """
    if isinstance(metadata, dict) and "columnas_entrada" in metadata:
        return list(metadata["columnas_entrada"])

    if hasattr(model, "feature_names_in_"):
        return list(model.feature_names_in_)

    return FEATURES.copy()


def normalize_prediction_label(raw_label, metadata=None):
    """
    Convierte la salida real del modelo a:
    - vulnerabilidad_baja
    - vulnerabilidad_alta

    Corrige el caso del .pkl donde:
    1 = VULNERABILIDAD_BAJA
    2 = VULNERABILIDAD_ALTA
    """
    # 1) Intentar usar el diccionario de clases guardado en el .pkl
    if isinstance(metadata, dict):
        class_map = metadata.get("clases", {})
        if isinstance(class_map, dict):
            # Probar con la llave original, int, str y float convertido
            candidates = [raw_label, str(raw_label)]
            try:
                candidates.append(int(raw_label))
            except Exception:
                pass
            try:
                candidates.append(float(raw_label))
            except Exception:
                pass

            for key in candidates:
                if key in class_map:
                    mapped = str(class_map[key]).lower().strip()
                    if "baja" in mapped:
                        return "vulnerabilidad_baja"
                    if "alta" in mapped:
                        return "vulnerabilidad_alta"

    # 2) Normalización por texto
    text = str(raw_label).lower().strip()

    if "baja" in text or "bajo" in text:
        return "vulnerabilidad_baja"
    if "alta" in text or "alto" in text:
        return "vulnerabilidad_alta"

    # 3) Fallback para codificaciones frecuentes
    # En este proyecto, si no hay metadatos y aparece 1/2, asumimos 1=baja y 2=alta.
    if text in ["1", "1.0"]:
        return "vulnerabilidad_baja"
    if text in ["2", "2.0"]:
        return "vulnerabilidad_alta"

    # En modelos binarios genéricos 0/1 suele ser 0=baja y 1=alta.
    if text in ["0", "0.0"]:
        return "vulnerabilidad_baja"

    return text


EXPECTED_MODEL_COLUMNS = get_expected_model_columns(model, model_metadata)
print("Columnas esperadas por el modelo:", EXPECTED_MODEL_COLUMNS)


def build_input_dataframe(
    genero,
    edad,
    discapacidad,
    cabeza_familia,
    adulto_mayor,
    grupo_etnico,
    seguridad_salud,
    nivel_educativo,
    condicion_ocupacional,
    barrio_vereda,
    zona_residencia,
):
    row = {
        "GENERO": genero,
        "EDAD": edad,
        "PERSONAS_EN_CONDICION_DE_DISCAPACIDAD": discapacidad,
        "HOMBRES_Y_MUJERES_CABEZA_DE_FAMILIA": cabeza_familia,
        "ADULTO_MAYOR": adulto_mayor,
        "GRUPOS_ETNICOS_AFRO_INDIGENA": grupo_etnico,
        "TIPO_DE_SEGURIDAD_EN_SALUD": seguridad_salud,
        "NIVEL_EDUCATIVO_QUE_TIENE_O_CURSA": nivel_educativo,
        "CONDICION_OCUPACIONAL": condicion_ocupacional,
        "BARRIO_O_VEREDA_DE_RESIDENCIA": barrio_vereda,
        "ZONA_DE_RESIDENCIA": zona_residencia,
    }

    input_df = pd.DataFrame([row])
    input_df = normalize_text_values(input_df)
    input_df["EDAD"] = pd.to_numeric(input_df["EDAD"], errors="coerce")

    # Si el modelo guardado fue entrenado con RANGO_EDAD, lo reconstruimos automáticamente.
    if "RANGO_EDAD" in EXPECTED_MODEL_COLUMNS:
        input_df["RANGO_EDAD"] = input_df["EDAD"].apply(make_age_range)

    # Asegurar que estén todas las columnas que el modelo espera.
    for col in EXPECTED_MODEL_COLUMNS:
        if col not in input_df.columns:
            input_df[col] = "SIN INFORMACION"

    return input_df[EXPECTED_MODEL_COLUMNS]


def validate_input(input_df):
    warnings_list = []

    edad = input_df.loc[0, "EDAD"]
    if pd.isna(edad):
        warnings_list.append("La edad ingresada no es válida. Debe ser un número.")
    elif edad < AGE_MIN or edad > AGE_MAX:
        warnings_list.append(
            f"La edad ingresada ({edad}) está fuera del rango observado en el dataset: {AGE_MIN} a {AGE_MAX} años."
        )

    # Validar solo columnas categóricas que existan en los catálogos del dataset.
    for col in ONEHOT_FEATURES + ORDINAL_FEATURES:
        if col not in input_df.columns:
            continue
        value = str(input_df.loc[0, col])
        known_values = set(CATEGORY_OPTIONS.get(col, []))
        if value not in known_values:
            warnings_list.append(
                f"Valor no visto en el dataset para {col}: '{value}'. El modelo puede manejarlo, pero la predicción es menos confiable."
            )

    return warnings_list


def get_prediction_probabilities(model, input_df, predicted_label):
    """
    Obtiene probabilidades y las asigna a la clase correcta.

    Esta función corrige el error principal del despliegue:
    no se debe asumir que clase 1 = alta.
    En este proyecto, según el .pkl:
    1 = vulnerabilidad_baja
    2 = vulnerabilidad_alta
    """
    probs = {
        "vulnerabilidad_baja": 0.0,
        "vulnerabilidad_alta": 0.0
    }

    if hasattr(model, "predict_proba"):
        try:
            raw_probs = model.predict_proba(input_df)[0]
            classes = list(model.classes_) if hasattr(model, "classes_") else []

            for cls, prob in zip(classes, raw_probs):
                label = normalize_prediction_label(cls, model_metadata)
                if label in probs:
                    probs[label] += float(prob)

            return probs
        except Exception as e:
            print("No se pudo calcular predict_proba:", repr(e))

    # Fallback si no hay predict_proba.
    label = normalize_prediction_label(predicted_label, model_metadata)
    if label in probs:
        probs[label] = 1.0

    return probs


def make_gray_probability_plot(probs):
    labels = ["Baja", "Alta"]
    values = [
        probs.get("vulnerabilidad_baja", 0),
        probs.get("vulnerabilidad_alta", 0)
    ]

    fig, ax = plt.subplots(figsize=(6.8, 4.2), constrained_layout=True)

    bars = ax.bar(
        labels,
        values,
        color=["#C9C9C9", "#4D4D4D"],
        edgecolor="#222222",
        linewidth=0.8
    )

    # Subimos el límite superior para que el porcentaje no se monte sobre el título.
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Probabilidad estimada", fontsize=11, labelpad=8)
    ax.set_xlabel("Clase estimada", fontsize=11, labelpad=8)
    ax.set_title("Resultado del simulador", fontsize=13, fontweight="bold", pad=18)
    ax.grid(axis="y", alpha=0.22)
    ax.spines[["top", "right"]].set_visible(False)

    for bar, value in zip(bars, values):
        y_pos = min(value + 0.035, 1.06)
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            y_pos,
            f"{value:.1%}",
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold"
        )

    return fig


def predict_profile(
    genero,
    edad,
    discapacidad,
    cabeza_familia,
    adulto_mayor,
    grupo_etnico,
    seguridad_salud,
    nivel_educativo,
    condicion_ocupacional,
    barrio_vereda,
    zona_residencia,
):
    try:
        input_df = build_input_dataframe(
            genero,
            edad,
            discapacidad,
            cabeza_familia,
            adulto_mayor,
            grupo_etnico,
            seguridad_salud,
            nivel_educativo,
            condicion_ocupacional,
            barrio_vereda,
            zona_residencia,
        )

        warnings_list = validate_input(input_df)

        raw_pred = model.predict(input_df)[0]
        pred = normalize_prediction_label(raw_pred, model_metadata)

        probs = get_prediction_probabilities(model, input_df, pred)
        fig = make_gray_probability_plot(probs)

        if pred == "vulnerabilidad_alta":
            main_result = "### Resultado: Vulnerabilidad alta"
            interpretation = (
                "El perfil ingresado fue clasificado como **vulnerabilidad_alta**. "
                "Este resultado es una estimación académica basada en patrones del dataset, no una decisión institucional."
            )
        elif pred == "vulnerabilidad_baja":
            main_result = "### Resultado: Vulnerabilidad baja"
            interpretation = (
                "El perfil ingresado fue clasificado como **vulnerabilidad_baja**. "
                "Este resultado es una estimación académica basada en patrones del dataset, no una decisión institucional."
            )
        else:
            main_result = f"### Resultado: {pred}"
            interpretation = (
                "El modelo devolvió una clase no estándar. Revisa el mapeo de clases del archivo `.pkl`."
            )

        prob_text = (
            f"\n\n**Probabilidad estimada baja:** {probs.get('vulnerabilidad_baja', 0):.1%}  "
            f"\n**Probabilidad estimada alta:** {probs.get('vulnerabilidad_alta', 0):.1%}"
        )

        debug_text = (
            f"\n\n<details><summary>Ver detalle técnico de depuración</summary>\n\n"
            f"- Salida cruda del modelo: `{raw_pred}`\n"
            f"- Clase interpretada por la app: `{pred}`\n"
            f"- Columnas usadas para predecir: `{list(input_df.columns)}`\n"
            f"</details>"
        )

        warning_text = ""
        if warnings_list:
            warning_text = "\n\n### Advertencias\n" + "\n".join([f"- {w}" for w in warnings_list])

        ethical_note = (
            "\n\n> **Nota:** esta app es académica y exploratoria. No debe usarse para asignar beneficios, "
            "priorizar población o tomar decisiones sociales reales sin validación adicional."
        )

        output_text = main_result + "\n\n" + interpretation + prob_text + warning_text + ethical_note + debug_text
        return output_text, fig

    except Exception as e:
        fig = make_gray_probability_plot({"vulnerabilidad_baja": 0, "vulnerabilidad_alta": 0})
        return f"### Error en la predicción\nRevisa los datos ingresados. Detalle técnico: `{repr(e)}`", fig

print("Catálogos, mapeo de clases y función de predicción corregidos.")
print("Rango de edad observado:", AGE_MIN, "a", AGE_MAX)


Columnas esperadas por el modelo: ['GENERO', 'EDAD', 'PERSONAS_EN_CONDICION_DE_DISCAPACIDAD', 'HOMBRES_Y_MUJERES_CABEZA_DE_FAMILIA', 'ADULTO_MAYOR', 'GRUPOS_ETNICOS_AFRO_INDIGENA', 'TIPO_DE_SEGURIDAD_EN_SALUD', 'NIVEL_EDUCATIVO_QUE_TIENE_O_CURSA', 'CONDICION_OCUPACIONAL', 'BARRIO_O_VEREDA_DE_RESIDENCIA', 'ZONA_DE_RESIDENCIA', 'RANGO_EDAD']
Catálogos, mapeo de clases y función de predicción corregidos.
Rango de edad observado: 16 a 85


## 8. Crear base de conocimiento y chatbot del proyecto

El chatbot está restringido al proyecto. Si la pregunta no está relacionada, devuelve una respuesta estándar.

In [55]:
def extract_pdf_text(path):
    if not os.path.exists(path):
        return ""
    try:
        from pypdf import PdfReader
        reader = PdfReader(path)
        pages = []
        for page in reader.pages:
            text = page.extract_text() or ""
            pages.append(text)
        return "\n".join(pages)
    except Exception as e:
        print("No se pudo leer el PDF:", repr(e))
        return ""

pdf_text = extract_pdf_text(PDF_PATH)

# Resumen base. Se usa aunque el PDF no esté disponible.
BASE_KNOWLEDGE = f"""
Proyecto: Índice de Vulnerabilidad Laboral en Pereira.
Tipo de problema: clasificación supervisada binaria.
Variable objetivo usada por la app: ETIQUETA_VULNERABILIDAD.
Clases: vulnerabilidad_baja y vulnerabilidad_alta.
Regla de construcción en la app: corte usando la mediana de PUNTAJE_VULNERABILIDAD.
Mediana calculada en el CSV cargado: {median_score}.
PUNTAJE_VULNERABILIDAD no se usa como variable predictora para evitar fuga de información.
Variables predictoras: {', '.join(FEATURES)}.
Preprocesamiento: OrdinalEncoder para nivel educativo; OneHotEncoder para variables nominales; EDAD pasa como variable numérica.
Modelo principal esperado: Random Forest o pipeline cargado desde archivo .pkl.
Estado de carga del modelo: {load_status}
Propósito: académico y exploratorio. No es una herramienta real de decisión social o política pública.
Limitaciones: depende de la calidad del dataset, puede reflejar sesgos de registro, categorías no vistas reducen confiabilidad y requiere validación institucional si se quisiera usar fuera del aula.
"""

try:
    metrics_df = pd.read_csv(METRICS_PATH) if os.path.exists(METRICS_PATH) else pd.DataFrame()
    metrics_text = metrics_df.to_string(index=False) if not metrics_df.empty else ""
except Exception:
    metrics_text = ""

KNOWLEDGE_TEXT = BASE_KNOWLEDGE + "\n\n" + metrics_text + "\n\n" + pdf_text[:20000]

PROJECT_KEYWORDS = [
    "proyecto", "objetivo", "base", "datos", "dataset", "variables", "variable", "etiqueta",
    "vulnerabilidad", "pereira", "modelo", "machine learning", "random forest", "accuracy",
    "precisión", "precision", "recall", "f1", "matriz", "confusión", "confusion", "preprocesamiento",
    "onehot", "ordinal", "fuga", "información", "informacion", "limitaciones", "ética", "etica",
    "sesgo", "trabajos futuros", "presentar", "presentación", "presentacion", "mediana",
    "puntaje", "clasificación", "clasificacion", "salud", "educativo", "ocupacional", "barrio",
    "gradio", "simulador", "aplicaciones"
]

OUT_OF_SCOPE = (
    "Lo siento, no puedo responder esa pregunta porque no está relacionada con este proyecto. "
    "Puedes preguntarme sobre la base de datos, el modelo, las variables, los resultados o las limitaciones."
)


def is_project_related(question):
    q = question.lower()
    return any(k in q for k in PROJECT_KEYWORDS)


def find_relevant_context(question, knowledge_text, max_chunks=4):
    """Búsqueda simple por coincidencia de palabras clave dentro del texto del proyecto."""
    q_words = [w for w in re.findall(r"\w+", question.lower()) if len(w) > 3]
    paragraphs = re.split(r"\n{2,}|(?<=\.)\s+", knowledge_text)
    scored = []
    for p in paragraphs:
        p_clean = p.strip()
        if len(p_clean) < 40:
            continue
        p_lower = p_clean.lower()
        score = sum(1 for w in q_words if w in p_lower)
        if score > 0:
            scored.append((score, p_clean))
    scored = sorted(scored, key=lambda x: x[0], reverse=True)
    return "\n\n".join([p for _, p in scored[:max_chunks]])


def project_chatbot(message, history):
    if not message or not message.strip():
        return "Escribe una pregunta sobre el proyecto."

    if not is_project_related(message):
        return OUT_OF_SCOPE

    q = message.lower()

    # Respuestas guiadas para preguntas frecuentes.
    if any(keyword in q for keyword in ["objetivo", "proposito", "finalidad", "para que"]) and "proyecto" in q:
        return (
            "El objetivo principal de este proyecto académico es construir una aplicación web en Gradio "
            "con un chatbot restringido a preguntas sobre el proyecto y un simulador de clasificación "
            "que predice la vulnerabilidad de un perfil sociodemográfico. Busca explorar la aplicación "
            "de Machine Learning para la clasificación binaria de niveles de vulnerabilidad en Pereira, "
            "sirviendo como herramienta exploratoria y educativa."
        )

    if "fuga" in q or "leakage" in q:
        return (
            "La fuga de información ocurre cuando se usa como predictor una variable que contiene directa o indirectamente "
            "la respuesta que el modelo debe aprender. En este proyecto, `PUNTAJE_VULNERABILIDAD` no debe usarse como "
            "variable predictora porque de él se construye `ETIQUETA_VULNERABILIDAD`. Si se incluyera, el modelo tendría "
            "una ventaja artificial y las métricas quedarían infladas."
        )

    if "variables" in q or "entrada" in q or "predictoras" in q:
        variable_details = []
        for feature in FEATURES:
            if feature in NUMERIC_FEATURES:
                variable_details.append(f"`{feature}` (numérica)")
            elif feature in ORDINAL_FEATURES:
                variable_details.append(f"`{feature}` (categórica ordinal con `OrdinalEncoder`)")
            elif feature in ONEHOT_FEATURES:
                variable_details.append(f"`{feature}` (categórica nominal con `OneHotEncoder`)")
            else:
                variable_details.append(f"`{feature}` (tipo no especificado, se asume categórica nominal)") # Fallback

        return (
            "Las variables de entrada (predictoras) usadas por el simulador son: "
            + ", ".join(variable_details)
            + ". `PUNTAJE_VULNERABILIDAD` se excluye para evitar fuga de información."
        )

    if "mediana" in q or "etiqueta" in q or "objetivo" in q:
        return (
            f"La variable objetivo de la app es `ETIQUETA_VULNERABILIDAD`, con dos clases: "
            f"`vulnerabilidad_baja` y `vulnerabilidad_alta`. En este notebook se reconstruye usando la mediana de "
            f"`PUNTAJE_VULNERABILIDAD`, que en el CSV cargado es {median_score}. Los perfiles con puntaje mayor "
            f"a la mediana se etiquetan como alta vulnerabilidad y los demás como baja vulnerabilidad."
        )

    if "accuracy" in q or "mejor modelo" in q or "random forest" in q or "modelo" in q:
        metric_line = ""
        if metrics_text:
            metric_line = "\n\nMétricas cargadas desde `metricas_modelo.csv`:\n" + metrics_text
        return (
            "El proyecto compara modelos de clasificación supervisada. Según las métricas del proyecto, "
            "Random Forest fue el mejor modelo reportado, con accuracy cercano a 98.96%. "
            "En la app, primero se intenta cargar el `.pkl`; si falla, se reentrena un pipeline con Random Forest desde el CSV."
            + metric_line
        )

    if "matriz" in q or "confusión" in q or "confusion" in q:
        return (
            "La matriz de confusión permite ver aciertos y errores por clase. En el informe del proyecto, "
            "el Random Forest reporta 1,337 casos de baja vulnerabilidad bien clasificados, 1,432 de alta vulnerabilidad "
            "bien clasificados, 9 bajas clasificadas como altas y 20 altas clasificadas como bajas. "
            "El error más delicado desde una perspectiva social sería clasificar una persona de alta vulnerabilidad como baja."
        )

    if "ética" in q or "etica" in q or "sesgo" in q or "limitaciones" in q:
        return (
            "Las principales limitaciones son: dependencia de la calidad del dataset, posible sesgo en los registros, "
            "categorías no vistas por el modelo, interpretación limitada de variables sociales complejas y riesgo de usar "
            "el resultado como decisión automática. Éticamente, la app debe presentarse como académica y exploratoria, "
            "no como una herramienta oficial para asignar ayudas o tomar decisiones de política pública."
        )

    if "present" in q or "exposición" in q or "exposicion" in q:
        return (
            "Para presentar el proyecto puedes seguir esta estructura: 1) problema social y contexto de Pereira; "
            "2) descripción de la base de datos; 3) construcción de la etiqueta de vulnerabilidad; "
            "4) limpieza y preprocesamiento; 5) modelos entrenados y métricas; 6) simulador en Gradio; "
            "7) limitaciones, ética y trabajos futuros. Es importante aclarar que es un ejercicio académico."
        )

    context = find_relevant_context(message, KNOWLEDGE_TEXT)
    if context:
        return (
            "Con base en la información del proyecto, esto es lo más relevante:\n\n"
            + context[:2500]
            + "\n\nRecuerda que la app tiene propósito académico y exploratorio."
        )

    return OUT_OF_SCOPE

print("Chatbot listo. Tamaño de la base de conocimiento:", len(KNOWLEDGE_TEXT), "caracteres")

Chatbot listo. Tamaño de la base de conocimiento: 22481 caracteres


## 9. Construir la interfaz en Gradio Blocks

La interfaz tiene tres pestañas: chatbot, simulador y guía de uso. Al final se ejecuta `demo.launch(share=True)` para obtener un enlace público temporal.

In [60]:
CUSTOM_CSS = """
.gradio-container {
    font-family: Inter, Arial, sans-serif !important;
    background: black !important;
    color: #F5F5DC !important; /* Creamy white for general text */
    font-size: 1.05rem !important;
    line-height: 1.6 !important;
}
.main-title {
    text-align: center;
    padding: 24px;
    border-radius: 20px;
    background: #0A2B42; /* Deeper, darker blue background for title box */
    color: #F5F5DC; /* Creamy white text color for main title div */
    margin-bottom: 18px;
    font-weight: 700; /* Ensure title is bold */
    box-shadow: 0 4px 12px rgba(0,0,0,0.4); /* Add a subtle shadow to the title box */
}
/* Explicitly set color for h1 and p within main-title to override other rules */
h1, .main-title h1,
p, .main-title p {
    color: #F5F5DC !important;
}
.main-title h1 {
    font-size: 2.8rem !important; /* Larger font size for main title */
    text-shadow: 2px 2px 4px rgba(0,0,0,0.5); /* Add text shadow for prominence */
}
.main-title p {
    font-size: 1.2rem !important; /* Slightly larger font for subtitle */
    text-shadow: 1px 1px 2px rgba(0,0,0,0.3); /* Add text shadow for prominence */
}
.card {
    background: #333333; /* Darker background for cards */
    border: 1px solid #444444; /* Slightly lighter border for cards */
    border-radius: 18px;
    padding: 18px;
    box-shadow: 0 1px 8px rgba(0,0,0,0.15); /* Softer shadow for cards */
    color: #F5F5DC !important;
}
.small-note {
    color: #AAAAAA; /* Slightly lighter grey for small notes */
    font-size: 0.92rem;
    line-height: 1.4;;
}
.warning-box {
    background: #444444; /* Darker background for warning */
    border-left: 5px solid #4169E1;
    padding: 12px;
    border-radius: 10px;
    color: #F5F5DC !important;
}

/* Botón principal en tonos de azul */
button.primary, .gr-button-primary {
    background: #2E5CBE !important; /* A slightly different shade of royal blue */
    border-color: #2E5CBE !important;
    color: white !important;
    transition: background-color 0.2s ease, box-shadow 0.2s ease;
    box-shadow: 0 2px 6px rgba(0,0,0,0.1);
}
button.primary:hover, .gr-button-primary:hover {
    background: #244AA6 !important; /* Darker on hover */
    border-color: #244AA6 !important;
    box-shadow: 0 4px 10px rgba(0,0,0,0.15);
}

/* Styling for tabs */
.gradio-tabs {
    border-radius: 15px;
    overflow: hidden;
    border: 1px solid #444444; /* Subtle border to entire tab component */
}

.gr-tabs-nav {
    background-color: #222222; /* Very dark grey for tab navigation */
    border-radius: 15px 15px 0 0; /* Rounded corners only at top */
    padding: 5px;
    box-shadow: none;
    border-bottom: 1px solid #444444; /* Subtle separator */
}

.gr-tabs-nav button {
    border-radius: 10px !important;
    margin: 3px !important;
    padding: 8px 15px !important;
    background-color: #555555 !important; /* Darker grey for default tab background */
    color: #F5F5DC !important; /* Creamy white for default tab text */
    font-weight: 400; /* Normal weight for non-selected tabs */
    border: none !important;
    box-shadow: none;
    transition: all 0.2s ease;
}
/* Target span inside button for extra specificity if needed */
.gr-tabs-nav button span {
    color: #F5F5DC !important;
}

.gr-tabs-nav button:hover {
    background-color: #666666 !important; /* Darker on hover for unselected */
    color: #F5F5DC !important;
}
.gr-tabs-nav button:hover span {
    color: #F5F5DC !important;
}

.gr-tabs-nav button.selected {
    background-color: #4169E1 !important; /* Royal blue for selected tab */
    color: white !important;
    box-shadow: 0 2px 8px rgba(0,0,0,0.08); /* Slight shadow for selected tab */
    font-weight: 600; /* Slightly bolder for selected tab */
}
.gr-tabs-nav button.selected span {
    color: white !important;
}

.gr-tab-content {
    background: #333333; /* Darker background for tab content */
    border: none;
    border-radius: 0 0 15px 15px;
    padding: 20px;
    box-shadow: 0 3px 10px rgba(0,0,0,0.05); /* Softer content shadow */
    color: #F5F5DC !important;
}

/* Ensure headings maintain the royal blue color or a suitable dark color */
h1, h2, h3, h4, h5, h6 {
    color: #F5F5DC !important; /* Creamy white for headings */
    font-weight: 600; /* Consistent heading weight */
    line-height: 1.3;
}

p, li, b, strong {
    color: #F5F5DC !important; /* Ensure paragraph, list, and bold text is creamy white */
}

a {
    color: #4169E1 !important; /* Keep links royal blue */
    text-decoration: none;
}
a:hover {
    text-decoration: underline;
}

/* Adjust Gradio specific components for a cleaner look */
.gradio-input, .gradio-dropdown, .gradio-slider {
    border-radius: 8px; /* Slightly rounded corners for inputs */
    border-color: #555555; /* Darker border for inputs */
    background-color: #222222; /* Dark background for inputs */
    box-shadow: 0 1px 3px rgba(0,0,0,0.1); /* Subtle shadow for inputs */
    color: #F5F5DC !important;
}

.gradio-label {
    font-weight: 500; /* Labels slightly bolder */
    color: #F5F5DC !important; /* Creamy white for labels */
}

/* Specific styling for the chatbot input textbox */
.gr-textbox textarea, .gr-textbox input {
    background-color: #222222 !important; /* Dark background for input area */
    border-radius: 8px !important; /* Rounded corners */
    border: 1px solid #555555 !important; /* Dark border */
    box-shadow: 0 1px 3px rgba(0,0,0,0.1) !important; /* Subtle shadow */
    color: #F5F5DC !important; /* Ensure input text is creamy white */
}

/* Styling for the chatbot container and messages */
.gradio-chatbot {
    background-color: #333333 !important; /* Dark background for chat history area */
    border: 1px solid #444444;
    border-radius: 12px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.1);
    padding: 10px; /* Add some internal padding */
}

/* Styling for message bubbles */
.gr-message {
    font-size: 0.95rem !important; /* Ensure message text is legible */
    line-height: 1.5 !important;
    color: #F5F5DC !important; /* Creamy white text for all messages */
    border-radius: 12px !important;
    padding: 10px 15px !important;
    margin-bottom: 8px !important;
    box-shadow: 0 1px 5px rgba(0,0,0,0.1); /* Subtle shadow for all messages */
}

.gr-message.bot {
    background-color: #1c3f60 !important; /* Dark blue background for bot messages */
    border: 1px solid #1c3f60; /* Dark border for bot messages */
    border-bottom-left-radius: 0px !important; /* Square bottom-left corner for bot */
}

.gr-message.user {
    background-color: #007bff !important; /* Blue background for user messages */
    border: 1px solid #007bff; /* Consistent blue border for user messages */
    border-bottom-right-radius: 0px !important; /* Square bottom-right corner for user */
}

.gradio-container .tab-nav button {
    background-color: #1c3f60 !important;
    color: white !important;
    border: none !important;
    border-radius: 10px 10px 0 0 !important;
    margin-right: 4px !important;
    font-weight: 600 !important;
    transition: all 0.2s ease-in-out;
}

/* Pestaña activa */
.gradio-container .tab-nav button.selected {
    background-color: #122c45 !important;  /* tono más oscuro */
    color: white !important;
    box-shadow: inset 0 -3px 0 rgba(255,255,255,0.15);
}

/* Hover */
.gradio-container .tab-nav button:hover {
    background-color: #163650 !important;
}

/* Styling for plot containers */
.plot-container {
    background-color: #333333; /* Darker background for the plot box */
    border: 1px solid #444444; /* Subtle border */
    border-radius: 15px; /* Rounded corners */
    padding: 15px;
    margin-bottom: 20px; /* Space between plot boxes */
    box-shadow: 0 2px 8px rgba(0,0,0,0.2); /* Soft shadow */
}
"""

example_questions = [
    "¿Cuál es el objetivo del proyecto?",
    "¿Qué variables usa el modelo?",
    "¿Por qué no se usa PUNTAJE_VULNERABILIDAD como predictor?",
    "¿Cuál fue el mejor modelo y qué accuracy obtuvo?",
    "¿Cómo interpreto la matriz de confusión?",
    "¿Qué limitaciones éticas tiene este proyecto?",
    "¿Cómo puedo presentar este proyecto final?",
]

with gr.Blocks(css=CUSTOM_CSS, title="Vulnerabilidad Pereira ML") as demo:
    gr.HTML("""
    <div class='main-title'>
        <h1>Índice de Vulnerabilidad en Pereira</h1>
        <p>Aplicación académica con Machine Learning, chatbot y simulador de clasificación</p>
    </div>
    """)

    with gr.Tab("Chatbot del proyecto"):
        gr.Markdown("## Asistente del proyecto")
        gr.Markdown("""
        ### Pregúntale al chatbot sobre el proyecto
        Puedes consultar sobre la base de datos, variables, objetivo, modelos, métricas, preprocesamiento, fuga de información, limitaciones, ética y presentación.
        """)
        chatbot = gr.ChatInterface(
            fn=project_chatbot,
            chatbot=gr.Chatbot(height=300, type="messages"), # Adjusted height
            textbox=
                gr.Textbox(
                placeholder="Ejemplo: ¿Por qué no se usa PUNTAJE_VULNERABILIDAD como predictor?",
                label="Pregunta"
            ),
            examples=example_questions,
            description="Responde únicamente preguntas relacionadas con el proyecto de vulnerabilidad en Pereira."
        )

    with gr.Tab("Simulador de clasificación"):
        gr.Markdown("""
        ### Ingresa un perfil sociodemográfico
        El sistema clasificará el perfil como `vulnerabilidad_baja` o `vulnerabilidad_alta`.
        Puedes seleccionar valores existentes o escribir valores nuevos. Si escribes categorías no vistas, aparecerá una advertencia.
        """)

        with gr.Row():
            with gr.Column(scale=1):
                genero = gr.Dropdown(CATEGORY_OPTIONS["GENERO"], label="Género", allow_custom_value=True)
                discapacidad = gr.Dropdown(CATEGORY_OPTIONS["PERSONAS_EN_CONDICION_DE_DISCAPACIDAD"], label="Personas en condición de discapacidad", allow_custom_value=True)
                cabeza_familia = gr.Dropdown(CATEGORY_OPTIONS["HOMBRES_Y_MUJERES_CABEZA_DE_FAMILIA"], label="Hombres y mujeres cabeza de familia", allow_custom_value=True)
                adulto_mayor = gr.Dropdown(CATEGORY_OPTIONS["ADULTO_MAYOR"], label="Adulto mayor", allow_custom_value=True)
                grupo_etnico = gr.Dropdown(CATEGORY_OPTIONS["GRUPOS_ETNICOS_AFRO_INDIGENA"], label="Grupos étnicos afro/indígena", allow_custom_value=True)

            with gr.Column(scale=1):
                seguridad_salud = gr.Dropdown(CATEGORY_OPTIONS["TIPO_DE_SEGURIDAD_EN_SALUD"], label="Tipo de seguridad en salud", allow_custom_value=True)
                nivel_educativo = gr.Dropdown(CATEGORY_OPTIONS["NIVEL_EDUCATIVO_QUE_TIENE_O_CURSA"], label="Nivel educativo que tiene o cursa", allow_custom_value=True)
                condicion_ocupacional = gr.Dropdown(CATEGORY_OPTIONS["CONDICION_OCUPACIONAL"], label="Condición ocupacional", allow_custom_value=True)
                barrio_vereda = gr.Dropdown(CATEGORY_OPTIONS["BARRIO_O_VEREDA_DE_RESIDENCIA"], label="Barrio o vereda de residencia", allow_custom_value=True)
                zona_residencia = gr.Dropdown(CATEGORY_OPTIONS["ZONA_DE_RESIDENCIA"], label="Zona de residencia", allow_custom_value=True)

        with gr.Row():
            edad = gr.Slider(minimum=AGE_MIN, maximum=AGE_MAX, value=30, step=1, label=f"Edad ({AGE_MIN}-{AGE_MAX} observado en el dataset)")

        predict_btn = gr.Button("Clasificar perfil", variant="primary")

        with gr.Row():
            result_text = gr.Markdown(label="Resultado")
            result_plot = gr.Plot(label="Visualización")

        predict_btn.click(
            fn=predict_profile,
            inputs=[
                genero,
                edad,
                discapacidad,
                cabeza_familia,
                adulto_mayor,
                grupo_etnico,
                seguridad_salud,
                nivel_educativo,
                condicion_ocupacional,
                barrio_vereda,
                zona_residencia,
            ],
            outputs=[result_text, result_plot]
        )

    with gr.Tab("Análisis Visual"):
        gr.Markdown("""
        ## Correlaciones y Análisis Visual
        Esta sección muestra visualizaciones clave para entender las relaciones en el dataset.
        """)

        # Crear una versión numérica de la variable objetivo para la correlación.
        # 0 para 'vulnerabilidad_baja', 1 para 'vulnerabilidad_alta'
        df_plot = df.copy()
        df_plot['ETIQUETA_VULNERABILIDAD_NUM'] = df_plot['ETIQUETA_VULNERABILIDAD'].map({'vulnerabilidad_baja': 0, 'vulnerabilidad_alta': 1})

        gr.Markdown("### Correlación de variables numéricas con la vulnerabilidad")
        with gr.Group(elem_classes="plot-container"):
            def plot_correlation_heatmap():
                fig, ax = plt.subplots(figsize=(8, 6))
                sns.heatmap(
                    df_plot[['EDAD', 'ETIQUETA_VULNERABILIDAD_NUM']].corr(),
                    annot=True,
                    cmap='coolwarm',
                    fmt=".2f",
                    linewidths=.5,
                    ax=ax
                )
                ax.set_title('Mapa de Calor de Correlación: Edad vs. Vulnerabilidad')
                return fig
            gr.Plot(plot_correlation_heatmap, label="Mapa de Calor de Correlación")

        gr.Markdown("### Distribución de Edad por Nivel de Vulnerabilidad")
        with gr.Group(elem_classes="plot-container"):
            def plot_age_vulnerability_boxplot():
                fig, ax = plt.subplots(figsize=(10, 6))
                sns.boxplot(
                    x='ETIQUETA_VULNERABILIDAD',
                    y='EDAD',
                    data=df_plot,
                    palette={'vulnerabilidad_baja': 'skyblue', 'vulnerabilidad_alta': 'salmon'},
                    ax=ax
                )
                ax.set_title('Distribución de Edad por Nivel de Vulnerabilidad')
                ax.set_xlabel('Nivel de Vulnerabilidad')
                ax.set_ylabel('Edad')
                return fig
            gr.Plot(plot_age_vulnerability_boxplot, label="Box Plot: Edad vs. Vulnerabilidad")

    with gr.Tab("Guía de uso"):
        gr.Markdown(f"""
        ## Guía rápida

        **1. Carga de datos**
        La app lee `dataset_definitivo.csv`. Si no existe `ETIQUETA_VULNERABILIDAD`, la reconstruye usando la mediana de `PUNTAJE_VULNERABILIDAD`.

        **2. Fuga de información**
        `PUNTAJE_VULNERABILIDAD` no se usa como variable predictora porque de allí se deriva la etiqueta.

        **3. Modelo**
        Primero se intenta cargar `modelo_Clasificacion_final.pkl`. Si falla, se entrena automáticamente un pipeline con Random Forest.

        **4. Preprocesamiento**
        - `NIVEL_EDUCATIVO_QUE_TIENE_O_CURSA`: OrdinalEncoder.
        - Variables nominales: OneHotEncoder con `handle_unknown='ignore'`.
        - `EDAD`: variable numérica.

        **5. Advertencias**
        Si escribes una categoría no vista o una edad fuera del rango `{AGE_MIN}-{AGE_MAX}`, la app mostrará una advertencia.

        **6. Uso responsable**
        La predicción es solo una estimación académica. No debe usarse para asignar beneficios, focalizar población ni reemplazar análisis social experto.

        ## Ejemplos de preguntas para el chatbot
        - ¿Cuál es el objetivo del proyecto?
        - ¿Qué variables usa el modelo?
        - ¿Cuál fue el mejor modelo?
        - ¿Qué significa la matriz de confusión?
        - ¿Qué limitaciones tiene el proyecto?
        - ¿Cómo presento este proyecto final?
        """)

    gr.Markdown(f"""
    <div class='warning-box'>
    <b>Nota importante:</b> esta aplicación es un prototipo académico y exploratorio. No debe usarse para tomar decisiones sociales reales.
    <br><b>Estado del modelo:</b> {load_status}
    </div>
    """)

# Ejecutar la app con enlace público temporal.
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e8f0b2cd4dbcf3eed2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
